# 7. AIND Ephys Results Collector

Build an `AINDEPhysResultsCollectorScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysResultsCollectorTask` for each coordinate. The task itself clones [aind-ephys-results-collector](https://github.com/AllenNeuralDynamics/aind-ephys-results-collector) on first use, seeds its `data/` with the union of every previous-stage output (plus a synthetic `ecephys_<session>/` folder so the capsule's `assert len(ecephys_sessions) == 1` passes), runs `code/run_capsule.py --process-name <name>`, and copies the unified layout (`preprocessed/`, `spikesorted/`, `postprocessed/`, `curated/`, `visualization/` + `processing.json` + `data_description.json`) into the single config's `coordinate_output_root` (`obi-output/07_aind_ephys_results_collector/grid_scan/0/`).

## Imports and prerequisites

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "aind-metadata-upgrader",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 35 packages in 6ms
Uninstalled 2 packages in 8ms
Installed 2 packages in 2ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'aind-metadata-upgrader'], returncode=0)

## Build the scan config

In [3]:
obi_output = (Path.cwd() / "../../../obi-output").resolve()
dispatch_output_path = obi_output / "01_aind_ephys_dispatch/grid_scan/0"
preprocessing_output_path = obi_output / "02_aind_ephys_preprocessing/grid_scan/0"
spikesort_output_path = obi_output / "03_aind_ephys_spikesort_kilosort4/grid_scan/0"
postprocessing_output_path = obi_output / "04_aind_ephys_postprocessing/grid_scan/0"
curation_output_path = obi_output / "05_aind_ephys_curation/grid_scan/0"
visualization_output_path = obi_output / "06_aind_ephys_visualization/grid_scan/0"

for p in (
    dispatch_output_path,
    preprocessing_output_path,
    spikesort_output_path,
    postprocessing_output_path,
    curation_output_path,
    visualization_output_path,
):
    assert p.exists(), f"{p} not found. Run earlier notebooks first."

scan_config = obi.AINDEPhysResultsCollectorScanConfig(
    initialize=obi.AINDEPhysResultsCollectorScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        preprocessing_output_path=preprocessing_output_path,
        spikesort_output_path=spikesort_output_path,
        postprocessing_output_path=postprocessing_output_path,
        curation_output_path=curation_output_path,
        visualization_output_path=visualization_output_path,
        process_name="sorted",
        session_name="ecephys_toy",
        subject_id="000000",
    ),
)

## Generate the grid scan and run the results-collector task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/07_aind_ephys_results_collector/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 19:09:50,969] INFO: Seeded data dir: /tmp/aind-ephys-results-collector/data


[2026-04-29 19:09:50,970] INFO: Running python -u run_capsule.py --process-name sorted


[2026-04-29 19:09:52,362] INFO: 

COLLECTING RESULTS
Loaded session name from JSON files: session0
Copying preprocessed folders to results:
	block0_None_recording1
Copying spikesorted folders to results:
	block0_None_recording1
Copying postprocessed and curated folders to results:
	block0_None_recording1
		Remapping recording path for postprocessed to ../results/postprocessed/block0_None_recording1.zarr
	Resolving path for folder_path - ../../../../../Users/james/Documents/obi/code/obi-main/obi-output/00_toy_example_recording
Copying visualization outputs to results:
Generating processing metadata
Generating data_description metadata
Failed to instantiate data description: 2 validation errors for DataDescription
funding_source.0.funder
  Input tag 'Allen Institute for Neural Dynamics' found using 'name' does not match any of the expected tags: 'Allen Institute', 'Chan Zuckerberg Initiative', 'MBF Bioscience', "Michael J. Fox Foundation for Parkinson's Research", 'National Center for Co

[PosixPath('../../../obi-output/07_aind_ephys_results_collector/grid_scan/0')]

## Inspect the collected processing.json

In [5]:
coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

processing = json.loads((coord_dir / "processing.json").read_text())
print("schema_version:", processing.get("schema_version"))
for pipeline in processing.get("pipelines", []):
    print("pipeline:", pipeline.get("name"))
process_names = [d.get("name") for d in processing.get("data_processes", [])]
print("data_processes:", process_names)

coordinate_output_root: ../../../obi-output/07_aind_ephys_results_collector/grid_scan/0
schema_version: 2.2.5
pipeline: AIND Ephys Pipeline
data_processes: ['Ephys preprocessing - block0_None_recording1', 'Spike sorting - block0_None_recording1', 'Ephys postprocessing - block0_None_recording1', 'Ephys curation - block0_None_recording1', 'Ephys visualization - block0_None_recording1']
